# Text Processing for All YouTube Transcript JSON Files

This notebook processes **all** raw video JSON files in `../data/raw` and writes timestamp-aware chunk files to `../data/processed`.

## Improvements included

- batch processing for **all** `video_*.json` files
- **timestamp-aware chunks**
- chunk schema now includes:
  - `start_time`
  - `end_time`
  - `start_time_str`
  - `end_time_str`
  - `source_segments`
- batch-level processing summary
- skip handling for files with missing or invalid transcripts

The raw JSON structure is compatible with this workflow because each transcript entry includes `text`, `start`, and `duration`.

In [1]:
import json
import os
import glob
from copy import deepcopy
from datetime import datetime
from statistics import mean
from typing import Any, Dict, List, Tuple

try:
    import pandas as pd
except ImportError:
    pd = None

## Configuration

Update these values only if you want different chunk sizes or output paths.

In [2]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

# Chunking controls
MAX_CHARS = 1200          # target upper bound for a chunk
MIN_CHARS = 300           # merge tiny trailing chunks when possible
OVERLAP_SEGMENTS = 2      # how many transcript segments to overlap between chunks
MAX_SOURCE_SEGMENTS = 80  # safety cap in case very short segments appear

os.makedirs(PROCESSED_DIR, exist_ok=True)

CHUNK_PREFIX = "chunks_timestamped_"
SUMMARY_FILENAME = "processing_summary_timestamped.json"

## Helper functions

In [3]:
def clean_text(text: str) -> str:
    if text is None:
        return ""
    return " ".join(str(text).replace("\n", " ").split()).strip()


def format_seconds(seconds: float) -> str:
    seconds = max(0, float(seconds))
    total = int(seconds)
    hours = total // 3600
    minutes = (total % 3600) // 60
    secs = total % 60
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def normalize_transcript_segments(transcript: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    normalized = []
    for idx, item in enumerate(transcript):
        text = clean_text(item.get("text", ""))
        if not text:
            continue

        start = float(item.get("start", 0.0) or 0.0)
        duration = float(item.get("duration", 0.0) or 0.0)
        end = start + max(duration, 0.0)

        normalized.append({
            "segment_index": idx,
            "text": text,
            "start": start,
            "duration": duration,
            "end": end,
            "char_count": len(text),
            "word_count": len(text.split()),
        })
    return normalized


def transcript_to_full_text(segments: List[Dict[str, Any]]) -> str:
    return " ".join(segment["text"] for segment in segments)


def build_chunk_from_segments(
    video_data: Dict[str, Any],
    normalized_segments: List[Dict[str, Any]],
    chunk_segments: List[Dict[str, Any]],
    chunk_index: int,
    total_chunks_placeholder: int = 0,
) -> Dict[str, Any]:
    metadata = video_data.get("metadata", {})
    video_id = video_data["video_id"]
    chunk_text = " ".join(segment["text"] for segment in chunk_segments).strip()

    start_time = chunk_segments[0]["start"]
    end_time = max(segment["end"] for segment in chunk_segments)

    description = metadata.get("description") or ""
    short_description = description[:240] + "..." if len(description) > 240 else description

    return {
        "chunk_id": f"{video_id}_chunk_{chunk_index:04d}",
        "video_id": video_id,
        "video_title": metadata.get("title"),
        "channel": metadata.get("channel"),
        "video_url": metadata.get("url"),
        "chunk_index": chunk_index,
        "total_chunks": total_chunks_placeholder,
        "text": chunk_text,
        "char_count": len(chunk_text),
        "word_count": len(chunk_text.split()),
        "start_time": round(start_time, 3),
        "end_time": round(end_time, 3),
        "start_time_str": format_seconds(start_time),
        "end_time_str": format_seconds(end_time),
        "duration_seconds": round(end_time - start_time, 3),
        "source_segments": [segment["segment_index"] for segment in chunk_segments],
        "source_segment_count": len(chunk_segments),
        "metadata": {
            "video_duration": metadata.get("duration"),
            "upload_date": metadata.get("upload_date"),
            "view_count": metadata.get("view_count"),
            "like_count": metadata.get("like_count"),
            "comment_count": metadata.get("comment_count"),
            "tags": metadata.get("tags") or [],
            "categories": metadata.get("categories") or [],
            "thumbnail": metadata.get("thumbnail"),
            "description": short_description,
            "transcript_language": video_data.get("transcript_language"),
            "transcript_status": video_data.get("transcript_status"),
            "playlist_index": metadata.get("playlist_index"),
        },
    }


def chunk_transcript_segments(
    video_data: Dict[str, Any],
    normalized_segments: List[Dict[str, Any]],
    max_chars: int = MAX_CHARS,
    min_chars: int = MIN_CHARS,
    overlap_segments: int = OVERLAP_SEGMENTS,
    max_source_segments: int = MAX_SOURCE_SEGMENTS,
) -> List[Dict[str, Any]]:
    if not normalized_segments:
        return []

    chunks = []
    start_idx = 0
    n = len(normalized_segments)

    while start_idx < n:
        current_segments = []
        current_chars = 0
        i = start_idx

        while i < n:
            seg = normalized_segments[i]
            proposed_chars = current_chars + (1 if current_segments else 0) + seg["char_count"]

            if current_segments and (
                proposed_chars > max_chars or len(current_segments) >= max_source_segments
            ):
                break

            current_segments.append(seg)
            current_chars = proposed_chars
            i += 1

        # Force at least one segment per chunk
        if not current_segments:
            current_segments.append(normalized_segments[start_idx])
            i = start_idx + 1

        # If the last chunk is too small, merge it into the previous chunk when practical
        if (
            chunks
            and i >= n
            and current_chars < min_chars
        ):
            previous = chunks[-1]
            merged_segments = previous["_raw_segments"] + current_segments
            merged_chunk = build_chunk_from_segments(
                video_data=video_data,
                normalized_segments=normalized_segments,
                chunk_segments=merged_segments,
                chunk_index=previous["chunk_index"],
            )
            merged_chunk["_raw_segments"] = merged_segments
            chunks[-1] = merged_chunk
            break

        chunk = build_chunk_from_segments(
            video_data=video_data,
            normalized_segments=normalized_segments,
            chunk_segments=current_segments,
            chunk_index=len(chunks),
        )
        chunk["_raw_segments"] = current_segments
        chunks.append(chunk)

        if i >= n:
            break

        next_start = max(i - overlap_segments, start_idx + 1)
        start_idx = next_start

    total_chunks = len(chunks)
    for chunk in chunks:
        chunk["total_chunks"] = total_chunks
        chunk.pop("_raw_segments", None)

    return chunks


def validate_chunks(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not chunks:
        return {
            "total_chunks": 0,
            "all_have_text": False,
            "all_have_ids": False,
            "all_have_timestamps": False,
            "all_have_source_segments": False,
            "min_char_count": 0,
            "max_char_count": 0,
            "avg_char_count": 0,
            "min_word_count": 0,
            "max_word_count": 0,
            "avg_word_count": 0,
        }

    char_counts = [chunk["char_count"] for chunk in chunks]
    word_counts = [chunk["word_count"] for chunk in chunks]

    return {
        "total_chunks": len(chunks),
        "all_have_text": all(bool(chunk.get("text")) for chunk in chunks),
        "all_have_ids": all(bool(chunk.get("chunk_id")) for chunk in chunks),
        "all_have_timestamps": all(
            chunk.get("start_time") is not None and chunk.get("end_time") is not None
            for chunk in chunks
        ),
        "all_have_source_segments": all(
            isinstance(chunk.get("source_segments"), list) and len(chunk["source_segments"]) > 0
            for chunk in chunks
        ),
        "min_char_count": min(char_counts),
        "max_char_count": max(char_counts),
        "avg_char_count": round(mean(char_counts), 2),
        "min_word_count": min(word_counts),
        "max_word_count": max(word_counts),
        "avg_word_count": round(mean(word_counts), 2),
    }


def process_video_file(
    json_file_path: str,
    processed_dir: str = PROCESSED_DIR,
    max_chars: int = MAX_CHARS,
    min_chars: int = MIN_CHARS,
    overlap_segments: int = OVERLAP_SEGMENTS,
) -> Dict[str, Any]:
    with open(json_file_path, "r", encoding="utf-8") as f:
        video_data = json.load(f)

    video_id = video_data["video_id"]
    metadata = video_data.get("metadata", {})
    transcript = video_data.get("transcript") or []
    transcript_status = video_data.get("transcript_status")

    if not transcript or transcript_status != "ok":
        return {
            "video_id": video_id,
            "video_title": metadata.get("title"),
            "status": "skipped_no_valid_transcript",
            "source_file": json_file_path,
            "output_file": None,
            "transcript_segments": len(transcript),
            "total_chunks": 0,
        }

    normalized_segments = normalize_transcript_segments(transcript)

    if not normalized_segments:
        return {
            "video_id": video_id,
            "video_title": metadata.get("title"),
            "status": "skipped_empty_transcript_after_cleaning",
            "source_file": json_file_path,
            "output_file": None,
            "transcript_segments": 0,
            "total_chunks": 0,
        }

    chunks = chunk_transcript_segments(
        video_data=video_data,
        normalized_segments=normalized_segments,
        max_chars=max_chars,
        min_chars=min_chars,
        overlap_segments=overlap_segments,
    )

    validation = validate_chunks(chunks)
    output_file = os.path.join(processed_dir, f"{CHUNK_PREFIX}{video_id}.json")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    full_text = transcript_to_full_text(normalized_segments)

    return {
        "video_id": video_id,
        "video_title": metadata.get("title"),
        "status": "processed",
        "source_file": json_file_path,
        "output_file": output_file,
        "transcript_segments": len(normalized_segments),
        "total_characters": len(full_text),
        "total_words": len(full_text.split()),
        "total_chunks": len(chunks),
        "first_chunk_start": chunks[0]["start_time"] if chunks else None,
        "last_chunk_end": chunks[-1]["end_time"] if chunks else None,
        "validation": validation,
    }

## Discover raw JSON files

In [4]:
all_json_files = sorted(glob.glob(os.path.join(RAW_DIR, "video_*.json")))
print(f"Found {len(all_json_files)} raw JSON file(s) in {RAW_DIR}")

all_json_files[:5]

Found 25 raw JSON file(s) in ../data/raw


['../data/raw\\video_04K0bLwCDdM.json',
 '../data/raw\\video_1YTKedLQOa0.json',
 '../data/raw\\video_21G7LA2DcGQ.json',
 '../data/raw\\video_78K0pbvHzjM.json',
 '../data/raw\\video_AkX6JqlWRqc.json']

In [ ]:
# Removes duplicates if any (should not be the case, but just in case)
all_json_files = sorted(set(glob.glob(os.path.join(RAW_DIR, "video_*.json"))))

## Process all raw files

This loop reads every `video_*.json` file from `../data/raw`, creates timestamp-aware chunks, and writes one processed JSON per video.

In [14]:
results = []

for i, json_file in enumerate(all_json_files, 1):
    print(f"[{i}/{len(all_json_files)}] Processing {os.path.basename(json_file)}")
    try:
        result = process_video_file(json_file)
        results.append(result)
        print(f"    -> {result['status']}")
    except Exception as e:
        error_result = {
            "video_id": None,
            "video_title": None,
            "status": f"failed: {e}",
            "source_file": json_file,
            "output_file": None,
            "transcript_segments": 0,
            "total_chunks": 0,
        }
        results.append(error_result)
        print(f"    -> failed: {e}")

[1/25] Processing video_04K0bLwCDdM.json
    -> processed
[2/25] Processing video_1YTKedLQOa0.json
    -> processed
[3/25] Processing video_21G7LA2DcGQ.json
    -> processed
[4/25] Processing video_78K0pbvHzjM.json
    -> processed
[5/25] Processing video_AkX6JqlWRqc.json
    -> processed
[6/25] Processing video_Bls5KnQOWkY.json
    -> processed
[7/25] Processing video_C-FEVzI8oe8.json
    -> processed
[8/25] Processing video_DLE-ieOVFjI.json
    -> processed
[9/25] Processing video_GHjopp47vvQ.json
    -> processed
[10/25] Processing video_Hn_iozUo9m4.json
    -> processed
[11/25] Processing video_JnYVz1TSmBQ.json
    -> processed
[12/25] Processing video_MvBqCeZllpQ.json
    -> processed
[13/25] Processing video_PaGJwOPg2kU.json
    -> processed
[14/25] Processing video_RB9mZbtcock.json
    -> processed
[15/25] Processing video_VRBpqM6ESrg.json
    -> processed
[16/25] Processing video_WSRqJdT2COE.json
    -> processed
[17/25] Processing video__DH3546mSCM.json
    -> processed
[18/25

## Batch summary

This summary fixes the earlier single-file logic by reporting **batch totals** across all processed JSON files.

In [ ]:
processed_results = [r for r in results if r["status"] == "processed"]
skipped_results = [r for r in results if r["status"].startswith("skipped")]
failed_results = [r for r in results if str(r["status"]).startswith("failed")]

summary = {
    "processed_at": datetime.now().isoformat(),
    "input_dir": RAW_DIR,
    "output_dir": PROCESSED_DIR,
    "chunking_config": {
        "max_chars": MAX_CHARS,
        "min_chars": MIN_CHARS,
        "overlap_segments": OVERLAP_SEGMENTS,
        "max_source_segments": MAX_SOURCE_SEGMENTS,
    },
    "totals": {
        "input_files": len(all_json_files),
        "processed": len(processed_results),
        "skipped": len(skipped_results),
        "failed": len(failed_results),
        "total_chunks_created": sum(r.get("total_chunks", 0) for r in processed_results),
        "total_transcript_segments": sum(r.get("transcript_segments", 0) for r in processed_results),
    },
    "files": results,
}

summary_path = os.path.join(PROCESSED_DIR, SUMMARY_FILENAME)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Done.")
print(f"Processed: {summary['totals']['processed']}")
print(f"Skipped:   {summary['totals']['skipped']}")
print(f"Failed:    {summary['totals']['failed']}")
print(f"Chunks:    {summary['totals']['total_chunks_created']}")
print(f"Summary:   {summary_path}")

NameError: name 'results' is not defined

In [16]:
for r in skipped_results:
    print(json.dumps(r, indent=2, ensure_ascii=False))

## Review summary as a table

In [17]:
if pd is not None:
    summary_df = pd.DataFrame(results)
    display(summary_df)
else:
    print("pandas is not installed; skipping table display.")
    for row in results[:10]:
        print(row)

,video_id,video_title,status,source_file,output_file,transcript_segments,total_characters,total_words,total_chunks,first_chunk_start,last_chunk_end,validation
0,04K0bLwCDdM,The Incredible Properties of Composite Materials,processed,../data/raw\video_04K0bLwCDdM.json,../data/processed\chunks_timestamped_04K0bLwCD...,224,19863,3159,20,0.180,1409.460,"{'total_chunks': 20, 'all_have_text': True, 'a..."
1,1YTKedLQOa0,Understanding Torsion,processed,../data/raw\video_1YTKedLQOa0.json,../data/processed\chunks_timestamped_1YTKedLQO...,132,8622,1579,9,0.540,605.840,"{'total_chunks': 9, 'all_have_text': True, 'al..."
2,21G7LA2DcGQ,Understanding Buckling,processed,../data/raw\video_21G7LA2DcGQ.json,../data/processed\chunks_timestamped_21G7LA2Dc...,183,12470,2072,12,1.359,886.041,"{'total_chunks': 12, 'all_have_text': True, 'a..."
3,78K0pbvHzjM,Understanding Plane Stress,processed,../data/raw\video_78K0pbvHzjM.json,../data/processed\chunks_timestamped_78K0pbvHz...,66,3580,620,4,0.030,247.400,"{'total_chunks': 4, 'all_have_text': True, 'al..."
4,AkX6JqlWRqc,Understanding True Stress and True Strain,processed,../data/raw\video_AkX6JqlWRqc.json,../data/processed\chunks_timestamped_AkX6JqlWR...,84,5655,932,6,0.179,400.919,"{'total_chunks': 6, 'all_have_text': True, 'al..."
5,Bls5KnQOWkY,Understanding the Area Moment of Inertia,processed,../data/raw\video_Bls5KnQOWkY.json,../data/processed\chunks_timestamped_Bls5KnQOW...,140,9407,1690,9,0.390,658.160,"{'total_chunks': 9, 'all_have_text': True, 'al..."
6,C-FEVzI8oe8,Understanding Shear Force and Bending Moment D...,processed,../data/raw\video_C-FEVzI8oe8.json,../data/processed\chunks_timestamped_C-FEVzI8o...,197,13407,2401,13,0.329,975.320,"{'total_chunks': 13, 'all_have_text': True, 'a..."
7,DLE-ieOVFjI,Understanding Young's Modulus,processed,../data/raw\video_DLE-ieOVFjI.json,../data/processed\chunks_timestamped_DLE-ieOVF...,88,5469,912,5,0.060,399.980,"{'total_chunks': 5, 'all_have_text': True, 'al..."
8,GHjopp47vvQ,Understanding the Finite Element Method,processed,../data/raw\video_GHjopp47vvQ.json,../data/processed\chunks_timestamped_GHjopp47v...,233,16115,2676,16,1.810,1105.712,"{'total_chunks': 16, 'all_have_text': True, 'a..."
9,Hn_iozUo9m4,Understanding and Analysing Trusses,processed,../data/raw\video_Hn_iozUo9m4.json,../data/processed\chunks_timestamped_Hn_iozUo9...,241,15468,2774,15,0.100,1052.480,"{'total_chunks': 15, 'all_have_text': True, 'a..."


In [18]:
#Check duplicate videos
print("Duplicate video_ids:", summary_df["video_id"].duplicated().sum())
print("Duplicate source files:", summary_df["source_file"].duplicated().sum())
print("Duplicate output files:", summary_df["output_file"].duplicated().sum())

Duplicate video_ids: 0
Duplicate source files: 0
Duplicate output files: 0


## Inspect one processed output file

This is useful for confirming that each chunk now includes:

- `start_time`
- `end_time`
- `start_time_str`
- `end_time_str`
- `source_segments`

In [19]:
processed_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, f"{CHUNK_PREFIX}*.json")))
print(f"Found {len(processed_files)} processed chunk file(s)")

if processed_files:
    sample_processed_file = processed_files[0]
    print("Sample file:", sample_processed_file)

    with open(sample_processed_file, "r", encoding="utf-8") as f:
        sample_chunks = json.load(f)

    print(f"Chunk count: {len(sample_chunks)}")
    if sample_chunks:
        sample_chunks[0]
else:
    print("No processed files found yet.")

Found 25 processed chunk file(s)
Sample file: ../data/processed\chunks_timestamped_04K0bLwCDdM.json
Chunk count: 20


## Inspect the specific example video

This cell tries to open the processed output for `video__DH3546mSCM.json` if it exists.

In [24]:
example_video_id = "mx7NYrUXDB0"
example_output_path = os.path.join(PROCESSED_DIR, f"{CHUNK_PREFIX}{example_video_id}.json")

if os.path.exists(example_output_path):
    with open(example_output_path, "r", encoding="utf-8") as f:
        example_chunks = json.load(f)

    print(f"Loaded {len(example_chunks)} chunk(s) for {example_video_id}")
    if example_chunks:
        print("First chunk:")
        print(json.dumps(example_chunks[0], indent=2, ensure_ascii=False)[:2500])
else:
    print(f"No processed file found for {example_video_id} yet.")
    print(f"Expected path: {example_output_path}")

Loaded 17 chunk(s) for mx7NYrUXDB0
First chunk:
{
  "chunk_id": "mx7NYrUXDB0_chunk_0000",
  "video_id": "mx7NYrUXDB0",
  "video_title": "Understanding Friction",
  "channel": "The Efficient Engineer",
  "video_url": "https://www.youtube.com/watch?v=mx7NYrUXDB0",
  "chunk_index": 0,
  "total_chunks": 17,
  "text": "Friction plays a role in every mechanical system. Whether it's gears meshing, pistons sliding, or bearings rotating, any time two surfaces contact each other, friction is there, resisting motion - whether you want it to or not. But what exactly is friction? What causes it, and how can we use it to our advantage? Let's find out. At the most basic level, friction is a resistive force that acts whenever surfaces are in contact and there's relative motion - or attempted motion - between them. Consider a block resting on a table. If you push on it with a small amount of force, it doesn't move, because a friction force develops at the interface between the two surfaces, exactly bal

## Optional: lightweight search over processed chunks

This is a simple string match helper so you can ask questions like:

- which chunk mentions **principal stress**
- what timestamp range does that chunk cover

For semantic retrieval later, you can embed these processed chunks directly.

In [26]:
def search_chunks(keyword: str, processed_dir: str = PROCESSED_DIR, limit: int = 10) -> List[Dict[str, Any]]:
    keyword_lower = keyword.lower().strip()
    matches = []

    if not keyword_lower:
        return matches

    processed_files = sorted(glob.glob(os.path.join(processed_dir, f"{CHUNK_PREFIX}*.json")))

    for path in processed_files:
        with open(path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            text = (chunk.get("text") or "").lower()
            if keyword_lower in text:
                matches.append({
                    "video_id": chunk.get("video_id"),
                    "video_title": chunk.get("video_title"),
                    "chunk_id": chunk.get("chunk_id"),
                    "start_time": chunk.get("start_time"),
                    "end_time": chunk.get("end_time"),
                    "start_time_str": chunk.get("start_time_str"),
                    "end_time_str": chunk.get("end_time_str"),
                    "text_preview": (chunk.get("text") or "")[:240] + "...",
                })

    return matches[:limit]

In [27]:
search_results = search_chunks("principal stress", limit=5)

if pd is not None and search_results:
    display(pd.DataFrame(search_results))
else:
    for row in search_results:
        print(row)

,video_id,video_title,chunk_id,start_time,end_time,start_time_str,end_time_str,text_preview
0,_DH3546mSCM,Understanding Stress Transformation and Mohr's...,_DH3546mSCM_chunk_0002,140.20,211.20,02:20,03:31,and that these maximum and minimum values are ...
1,_DH3546mSCM,Understanding Stress Transformation and Mohr's...,_DH3546mSCM_chunk_0004,277.32,349.20,04:37,05:49,Normal stresses are positive if they are tensi...
2,_DH3546mSCM,Understanding Stress Transformation and Mohr's...,_DH3546mSCM_chunk_0005,342.70,427.90,05:42,07:07,An important thing to note is that angles in M...
3,xkbQnBAOFEg,"Understanding Failure Theories (Tresca, von Mi...",xkbQnBAOFEg_chunk_0001,77.79,151.75,01:17,02:31,"failure theories, each of which we know works ..."
4,xkbQnBAOFEg,"Understanding Failure Theories (Tresca, von Mi...",xkbQnBAOFEg_chunk_0002,142.66,221.63,02:22,03:41,"Although it’s simple to apply, it’s not actual..."
